# Data Information

This dataset contains medical insurance cost information for 1338 individuals. It includes demographic and health-related variables such as age, sex, BMI, number of children, smoking status, and residential region in the US. The target variable is charges, which represents the medical insurance cost billed to the individual.

Source: https://doi.org/10.34740/kaggle/dsv/12853160

# Dataset

In [550]:
import pandas as pd

data_df = pd.read_csv("insurance.csv")
data_df.head(10)

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


# Problem Definition

## Objective

- use the dataset to estimate insurance bill/cost per person
- identify which variables are most associated with higher or lower insurance bill

## Usecase

Corporate Wellness Program
- Identify high-risk employees based on predicted healthcare costs
- Enable targeted interventions to reduce the company’s overall healthcare expenses

Insurance Company
- Analyze healthcare cost trends across different population segments
- Enable dynamic premium pricing based on individual risk profiles
- Provide personalized insurance plans tailored to each customer

Example Use Case
A person with certain conditions who is predicted to have high healthcare costs can be recommended lifestyle interventions

# Data Preparation

## 1. Data Understanding

In [551]:
print("Shape:", data_df.shape)

Shape: (1338, 7)


In [552]:
print("Columns:", data_df.columns)

Columns: Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='str')


In [553]:
data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [554]:
data_df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [555]:
data_df.duplicated().sum()

np.int64(1)

In [556]:
data_df = data_df.drop_duplicates()
data_df.duplicated().sum()

np.int64(0)

In [557]:
data_df.describe().round(2)

,age,bmi,children,charges
count,1337.00,1337.00,1337.00,1337.00
mean,39.22,30.66,1.10,13279.12
std,14.04,6.10,1.21,12110.36
min,18.00,15.96,0.00,1121.87
25%,27.00,26.29,0.00,4746.34
50%,39.00,30.40,1.00,9386.16
75%,51.00,34.70,2.00,16657.72
max,64.00,53.13,5.00,63770.43


In [558]:
data_df.describe(include="str")

,sex,smoker,region
count,1337,1337,1337
unique,2,2,4
top,male,no,southeast
freq,675,1063,364


# 2. Data Cleaning

## 2.1 Outlier 

In [559]:
num_cols = ['age', 'bmi', 'charges']

for col in num_cols:
    Q1 = data_df[col].quantile(0.25)
    Q3 = data_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = data_df[(data_df[col] < lower) | (data_df[col] > upper)]
    
    print(f"{col}: {len(outliers)} outliers")

age: 0 outliers
bmi: 9 outliers
charges: 139 outliers


The BMI outlier in this dataset is above ~47.3 (upper bound IQR), with a max of 53.13. This value is included in the Obesity Class III category, which medically actually exists and is documented. This is not a data entry error.

Outlier charges are retained because they are the target variable that the model must predict.

In [560]:
data_df['children'].value_counts()

children
0    573
1    324
2    240
3    157
4     25
5     18
Name: count, dtype: int64

In [561]:
data_df['region'].value_counts()

region
southeast    364
southwest    325
northwest    324
northeast    324
Name: count, dtype: int64

In [562]:
data_df['sex'].value_counts()

sex
male      675
female    662
Name: count, dtype: int64

In [563]:
data_df['smoker'].value_counts()

smoker
no     1063
yes     274
Name: count, dtype: int64

# 3. Data Transformation

## 3.1 Feature Encoding

In [564]:
data_df['gender_encoded'] = data_df['sex'].map({'male': 0, 'female': 1})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,gender_encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1
1,18,male,33.770,1,no,southeast,1725.55230,0
2,28,male,33.000,3,no,southeast,4449.46200,0
3,33,male,22.705,0,no,northwest,21984.47061,0
4,32,male,28.880,0,no,northwest,3866.85520,0


In [565]:
data_df['smoker_encoded'] = data_df['smoker'].map({'no': 0, 'yes': 1})
data_df.head(5)

,age,sex,bmi,children,smoker,region,charges,gender_encoded,smoker_encoded
0,19,female,27.900,0,yes,southwest,16884.92400,1,1
1,18,male,33.770,1,no,southeast,1725.55230,0,0
2,28,male,33.000,3,no,southeast,4449.46200,0,0
3,33,male,22.705,0,no,northwest,21984.47061,0,0
4,32,male,28.880,0,no,northwest,3866.85520,0,0


In [566]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, drop='first')  # drop='first' opsional

region_encoded = encoder.fit_transform(data_df[['region']])

region_columns = encoder.get_feature_names_out(['region'])
region_df = pd.DataFrame(region_encoded, columns=region_columns, index=data_df.index)

data_df = data_df.drop('region', axis=1)
data_df = pd.concat([data_df, region_df], axis=1)

data_df.head(5)

,age,sex,bmi,children,smoker,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
0,19,female,27.900,0,yes,16884.92400,1,1,0.0,0.0,1.0
1,18,male,33.770,1,no,1725.55230,0,0,0.0,1.0,0.0
2,28,male,33.000,3,no,4449.46200,0,0,0.0,1.0,0.0
3,33,male,22.705,0,no,21984.47061,0,0,1.0,0.0,0.0
4,32,male,28.880,0,no,3866.85520,0,0,1.0,0.0,0.0


In [567]:
data_df = data_df.drop(columns=['sex', 'smoker'])
data_df.head(5)

,age,bmi,children,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,1,1,0.0,0.0,1.0
1,18,33.770,1,1725.55230,0,0,0.0,1.0,0.0
2,28,33.000,3,4449.46200,0,0,0.0,1.0,0.0
3,33,22.705,0,21984.47061,0,0,1.0,0.0,0.0
4,32,28.880,0,3866.85520,0,0,1.0,0.0,0.0


# Data Analysis

## 1. Descriptive Statistics

In [568]:
data_df.describe().round(2)

,age,bmi,children,charges,gender_encoded,smoker_encoded,region_northwest,region_southeast,region_southwest
count,1337.00,1337.00,1337.00,1337.00,1337.0,1337.0,1337.00,1337.00,1337.00
mean,39.22,30.66,1.10,13279.12,0.5,0.2,0.24,0.27,0.24
std,14.04,6.10,1.21,12110.36,0.5,0.4,0.43,0.45,0.43
min,18.00,15.96,0.00,1121.87,0.0,0.0,0.00,0.00,0.00
25%,27.00,26.29,0.00,4746.34,0.0,0.0,0.00,0.00,0.00
50%,39.00,30.40,1.00,9386.16,0.0,0.0,0.00,0.00,0.00
75%,51.00,34.70,2.00,16657.72,1.0,0.0,0.00,1.00,0.00
max,64.00,53.13,5.00,63770.43,1.0,1.0,1.00,1.00,1.00
